In [1]:
pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model and preprocess function

In [4]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


### Prepare the few-shot cache

In [25]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [27]:
ds['test'][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=570x622>,
 'wnid': 'n02088094',
 'class_name': 'afghan_hound'}

In [28]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1].replace('_', ' ') for pair in unique_pairs]

In [29]:
len(r_class_names)

200

In [30]:
r_class_names[:5]

['goldfish', 'great white shark', 'hammerhead', 'stingray', 'hen']

In [31]:
wnid_to_r_index = {wnid: i for i, wnid in enumerate(r_wnids)}

for wnid, idx in list(wnid_to_r_index.items())[:5]:
  print(f"wnid: {wnid}, idx: {idx}")

wnid: n01443537, idx: 0
wnid: n01484850, idx: 1
wnid: n01494475, idx: 2
wnid: n01498041, idx: 3
wnid: n01514859, idx: 4


In [32]:
from collections import defaultdict

classes_to_indices = defaultdict(list)
all_indices = []

for idx in range(len(ds['test'])):
  label = ds['test'][idx]['class_name'].replace('_', ' ')
  classes_to_indices[label].append(idx)
  all_indices.append(idx)

In [36]:
print(len(classes_to_indices))
print(len(all_indices))

200
30000


In [37]:
import random

few_shot_indices = []

for cls, indices in classes_to_indices.items():
  sampled = random.sample(indices, 16)
  few_shot_indices.extend(sampled)

In [38]:
len(few_shot_indices) == (200 * 16)

True

In [39]:
class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, preprocess, wnid_to_index):
        self.hf_dataset = hf_dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        image = self.preprocess(example["image"].convert("RGB"))
        label = self.wnid_to_index[example["wnid"]]
        return image, label

In [40]:
full_wrapped = HFImageDataset(ds['test'], preprocess, wnid_to_r_index)

In [41]:
from torch.utils.data import Subset

raw_few_shot_ds = Subset(full_wrapped, few_shot_indices)
print(len(raw_few_shot_ds))

3200


In [42]:
eval_indices = list(set(all_indices) - set(few_shot_indices))
print(len(eval_indices))

26800


In [43]:
raw_eval_ds = Subset(full_wrapped, eval_indices)
print(len(raw_eval_ds))

26800


In [44]:
raw_few_shot_loader = DataLoader(raw_few_shot_ds, batch_size = 32, shuffle = True)
raw_eval_loader = DataLoader(raw_eval_ds, batch_size = 32)

### Build the cache model

In [45]:
from clip_zeroshot import build_and_cache_image_features, build_and_cache_text_features

In [46]:
few_shot_image_cache = build_and_cache_image_features(model, device, raw_few_shot_loader, './features', 'few_shot_image_features')

  0%|          | 0/100 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/few_shot_image_features.pt


In [47]:
few_shot_image_features = few_shot_image_cache['image_features']
few_shot_image_labels = few_shot_image_cache['labels']

In [48]:
print(few_shot_image_features.shape)
print(few_shot_image_labels.shape)

torch.Size([3200, 512])
torch.Size([3200])


In [51]:
few_shot_image_labels[:10]

tensor([ 56, 191, 122,   5, 175,  39, 135, 121,  23, 188])

In [49]:
import torch.nn.functional as F

one_hot = F.one_hot(few_shot_image_labels, num_classes=len(r_class_names))

In [50]:
print(one_hot.shape)

torch.Size([3200, 200])


In [61]:
cache_keys = few_shot_image_features
cache_values = one_hot.float()

### Build the zero shot classifier

In [62]:
from imagenet_classes import IMAGENET_TEMPLATES

In [65]:
text_features = build_and_cache_text_features(model = model, device= device, tokenizer = tokenizer, classnames = r_class_names, templates = IMAGENET_TEMPLATES, cache_dir='./features', file_name='text-features')

  0%|          | 0/200 [00:00<?, ?it/s]

Text features has been saved at ./features/text-features.pt


In [66]:
eval_cache = build_and_cache_image_features(model, device, raw_eval_loader, './features', 'r_eval_features')
eval_features = eval_cache['image_features']
eval_labels = eval_cache['labels']

  0%|          | 0/838 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/r_eval_features.pt
